# LeetCode #98 — Validate Binary Search Tree: truyền khoảng (lo, hi) hoặc duyệt inorder

**Mức độ:** Medium · **Chủ đề:** Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def isValidBST(self, root: Optional[TreeNode]) -> bool:
```

Mục tiêu của notebook:

1. Hiểu đúng định nghĩa BST — và cái bẫy "chỉ so với con trực tiếp".
2. Cách 1: DFS truyền **khoảng hợp lệ** `(lo, hi)` xuống từng nút.
3. Cách 2: **duyệt inorder** phải cho dãy **tăng nghiêm ngặt**.
4. Viết inorder **không đệ quy** bằng stack.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, kiểm tra nó có phải **BST hợp lệ** không. Theo đề, BST hợp lệ khi:

- Cây con **trái** của một nút chỉ chứa các nút có giá trị **nhỏ hơn hẳn** giá trị nút đó.
- Cây con **phải** chỉ chứa các nút có giá trị **lớn hơn hẳn**.
- Cả cây con trái và phải cũng phải là BST.

**Constraints (theo đề):**

- Số nút trong khoảng `[1, 10^4]`.
- `-2^31 <= Node.val <= 2^31 - 1`

---

## Ví dụ

```text
    2
   / \        ->  True
  1   3

    5
   / \
  1   4       ->  False   (4 nằm bên phải 5 nhưng 4 < 5; và 3 < 5)
     / \
    3   6
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Cây nhị phân tìm kiếm (BST)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [7]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [8]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.3 Áp dụng vào bài này

Kiểm tra **BST**: truyền xuống **khoảng cho phép** `(lo, hi)` cho mỗi nút, hoặc duyệt **inorder** và kiểm tra dãy tăng ngặt — đúng như phần "kiểm tra một cây có phải BST" ở trên (và tránh lỗi chỉ so với con trực tiếp).

# 3. Nền tảng: `TreeNode` và dựng cây từ danh sách level-order

LeetCode cho cây dạng `[5, 1, 4, null, null, 3, 6]` (duyệt theo tầng, `None` cho chỗ trống).

In [9]:
from typing import List, Optional
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def inorder_list(root: Optional[TreeNode]) -> list:
    return inorder_list(root.left) + [root.val] + inorder_list(root.right) if root else []

t = build_tree([5, 1, 4, None, None, 3, 6])
print(t.val, t.left.val, t.right.val, t.right.left.val, t.right.right.val)
print("inorder:", inorder_list(t))

5 1 4 3 6
inorder: [1, 5, 3, 4, 6]


# 4. Cái bẫy: chỉ so với con trực tiếp

Ý tưởng sai phổ biến: kiểm tra `left.val < node.val < right.val` ở **mọi nút**.

In [10]:
def wrong_local_check(root: Optional[TreeNode]) -> bool:
    if not root:
        return True
    if root.left and root.left.val >= root.val:
        return False
    if root.right and root.right.val <= root.val:
        return False
    return wrong_local_check(root.left) and wrong_local_check(root.right)

bad = build_tree([5, 4, 6, None, None, 3, 7])
print("Cây [5, 4, 6, null, null, 3, 7] — kiểm tra cục bộ nói:", wrong_local_check(bad))
print("inorder:", inorder_list(bad))

Cây [5, 4, 6, null, null, 3, 7] — kiểm tra cục bộ nói:

 True
inorder: [4, 5, 3, 6, 7]


```text
      5
     / \
    4   6
       / \
      3   7
```

Mỗi nút đều đúng so với con trực tiếp, nhưng nút `3` nằm trong cây con **phải** của `5` mà `3 < 5` → **không** phải BST. Ràng buộc BST là với **mọi tổ tiên**, không chỉ cha.

# 5. Cách 1: DFS truyền khoảng hợp lệ

Mỗi nút phải nằm trong một **khoảng mở** `(lo, hi)` do các tổ tiên quy định:

- Gốc: `(-∞, +∞)`.
- Đi sang **trái** của nút `v`: cận trên thành `v` → `(lo, v)`.
- Đi sang **phải**: cận dưới thành `v` → `(v, hi)`.

```text
valid(node, lo, hi):
    if node is None: return True
    if not (lo < node.val < hi): return False
    return valid(node.left, lo, node.val) and valid(node.right, node.val, hi)
```

In [11]:
def is_valid_range(root: Optional[TreeNode]) -> bool:
    def valid(node, lo, hi):
        if not node:
            return True
        if not (lo < node.val < hi):
            return False
        return valid(node.left, lo, node.val) and valid(node.right, node.val, hi)
    return valid(root, float("-inf"), float("inf"))

print(is_valid_range(build_tree([2, 1, 3])), is_valid_range(build_tree([5, 1, 4, None, None, 3, 6])), is_valid_range(bad))

True

 False False


`float("-inf")` / `float("inf")` dùng làm "không có cận". Giá trị nút có thể là `-2^31` hoặc `2^31 − 1` — nên **không** được dùng chính các số đó làm cận (Lỗi 2).

# 6. Mô phỏng: khoảng ở mỗi nút

In [12]:
def trace_range(values: list) -> bool:
    root = build_tree(values)
    def valid(node, lo, hi, depth):
        if not node:
            return True
        ok = lo < node.val < hi
        print("  " * depth + f"nút {node.val}: cần trong ({lo}, {hi}) -> {'OK' if ok else 'VI PHẠM'}")
        if not ok:
            return False
        return valid(node.left, lo, node.val, depth + 1) and valid(node.right, node.val, hi, depth + 1)
    res = valid(root, float("-inf"), float("inf"), 0)
    print("Kết quả:", res)
    return res

_ = trace_range([5, 4, 6, None, None, 3, 7])

nút 5: cần trong (-inf, inf) -> OK
  nút 4: cần trong (-inf, 5) -> OK
  nút 6: cần trong (5, inf) -> OK
    nút 3: cần trong (5, 6) -> VI PHẠM
Kết quả: False


Nút `3` được yêu cầu nằm trong `(5, 6)` — vì nó là con trái của `6` (cận trên 6) **và** nằm trong cây phải của `5` (cận dưới 5).

# 7. Cách 2: Inorder phải tăng nghiêm ngặt

Tính chất: cây nhị phân là BST ⇔ duyệt **inorder** (trái – gốc – phải) cho dãy **tăng nghiêm ngặt**.

Không cần lưu cả dãy: chỉ so mỗi giá trị với giá trị **trước đó** trong inorder. Viết bằng stack để tránh đệ quy sâu (cây lệch 10^4 nút sẽ vượt giới hạn đệ quy mặc định 1000 của Python).

```text
stack = [], node = root, prev = -∞
while stack or node:
    while node: stack.append(node); node = node.left     # đi hết về trái
    node = stack.pop()                                  # thăm
    if node.val <= prev: return False
    prev = node.val
    node = node.right                                   # sang phải
return True
```

In [13]:
def is_valid_inorder(root: Optional[TreeNode]) -> bool:
    stack, node, prev = [], root, float("-inf")
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        if node.val <= prev:
            return False
        prev = node.val
        node = node.right
    return True

print(is_valid_inorder(build_tree([2, 1, 3])), is_valid_inorder(build_tree([5, 1, 4, None, None, 3, 6])), is_valid_inorder(bad))

True False False


In [14]:
def trace_inorder(values: list) -> bool:
    root = build_tree(values)
    stack, node, prev = [], root, float("-inf")
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        verdict = "OK" if node.val > prev else "KHÔNG tăng -> False"
        print(f"  thăm {node.val} (trước đó {prev}) -> {verdict} | stack còn {[x.val for x in stack]}")
        if node.val <= prev:
            return False
        prev = node.val
        node = node.right
    print("  -> True")
    return True

_ = trace_inorder([5, 4, 6, None, None, 3, 7])

  thăm 4 (trước đó -inf) -> OK | stack còn [5]
  thăm 5 (trước đó 4) -> OK | stack còn []
  thăm 3 (trước đó 5) -> KHÔNG tăng -> False | stack còn [6]


# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def isValidBST(self, root: Optional[TreeNode]) -> bool:
        stack, node, prev = [], root, float("-inf")
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            if node.val <= prev:
                return False
            prev = node.val
            node = node.right
        return True

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


def brute_valid(root):
    """Định nghĩa trực tiếp: mọi nút ở trái < gốc < mọi nút ở phải, đệ quy."""
    if not root:
        return True
    return all(v < root.val for v in inorder_list(root.left)) and \
           all(v > root.val for v in inorder_list(root.right)) and \
           brute_valid(root.left) and brute_valid(root.right)


def random_tree(n, lo=-3, hi=3):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(lo, hi), random_tree(k, lo, hi), random_tree(n - 1 - k, lo, hi))


sol = Solution()
check("Ví dụ 1", sol.isValidBST(build_tree([2, 1, 3])), True)
check("Ví dụ 2", sol.isValidBST(build_tree([5, 1, 4, None, None, 3, 6])), False)
check("Bẫy tổ tiên", sol.isValidBST(bad), False)
check("Một nút", sol.isValidBST(build_tree([1])), True)
check("Trùng giá trị", sol.isValidBST(build_tree([2, 2, 2])), False)
check("Cận 32-bit", sol.isValidBST(build_tree([-2**31, None, 2**31 - 1])), True)
check("Chỉ một nút cực tiểu", sol.isValidBST(build_tree([-2**31])), True)

deep = None
for v in range(10**4, 0, -1):          # cây lệch phải 10^4 nút: 1 -> 2 -> ... (dựng từ dưới lên)
    deep = TreeNode(v, None, deep)
check("Cây lệch 10^4 nút", sol.isValidBST(deep), True)

random.seed(98)
for _ in range(5000):
    root = random_tree(random.randint(1, 7))
    expected = brute_valid(root)
    assert sol.isValidBST(root) == expected == is_valid_range(root), inorder_list(root)
print("5000 cây ngẫu nhiên: khớp định nghĩa trực tiếp")

Ví dụ 1                    PASS  True


Ví dụ 2                    PASS  False
Bẫy tổ tiên                PASS  False
Một nút                    PASS  True
Trùng giá trị              PASS  False
Cận 32-bit                 PASS  True
Chỉ một nút cực tiểu       PASS  True
Cây lệch 10^4 nút          PASS  True


5000 cây ngẫu nhiên: khớp định nghĩa trực tiếp


`is_valid_range` (đệ quy) sẽ gặp `RecursionError` với cây lệch 10^4 nút trong Python mặc định — bản stack thì không.

# 9. Vì sao đúng? Độ phức tạp?

**Cách khoảng:** bằng quy nạp, `(lo, hi)` truyền tới một nút chính là giao của mọi ràng buộc từ tổ tiên: nút nằm trong cây trái của tổ tiên `a` thì phải `< a.val`, trong cây phải thì `> a.val`. Kiểm tra mọi nút thỏa khoảng của nó ⇔ định nghĩa BST.

**Cách inorder:** inorder của BST liệt kê giá trị theo thứ tự tăng (cây trái, rồi gốc, rồi cây phải). Ngược lại, nếu inorder tăng nghiêm ngặt thì với mỗi nút, mọi nút cây trái (đứng trước nó) nhỏ hơn và mọi nút cây phải (đứng sau) lớn hơn.

```text
Time:  O(n)
Space: O(h) — h là chiều cao (stack đệ quy hoặc stack tự quản); xấu nhất O(n)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Chỉ so với con trực tiếp

Như mục 4.

## Lỗi 2: Dùng `-2^31` / `2^31 − 1` làm cận ban đầu

In [17]:
def wrong_int_bounds(root: Optional[TreeNode]) -> bool:
    def valid(node, lo, hi):
        if not node:
            return True
        if not (lo < node.val < hi):
            return False
        return valid(node.left, lo, node.val) and valid(node.right, node.val, hi)
    return valid(root, -2**31, 2**31 - 1)            # SAI: nút có thể bằng đúng các giá trị này

print("Sai :", wrong_int_bounds(build_tree([2**31 - 1])), "| Đúng:", sol.isValidBST(build_tree([2**31 - 1])))

Sai : False | Đúng: True


## Lỗi 3: Cho phép bằng (`<=`)

Đề yêu cầu **nhỏ hơn hẳn** / **lớn hơn hẳn**. `[2, 2, 2]` không hợp lệ.

In [18]:
def wrong_allow_equal(root: Optional[TreeNode]) -> bool:
    vals = inorder_list(root)
    return all(a <= b for a, b in zip(vals, vals[1:]))   # SAI: phải là a < b

print("Sai :", wrong_allow_equal(build_tree([2, 2, 2])), "| Đúng:", sol.isValidBST(build_tree([2, 2, 2])))

Sai : True | Đúng: False


## Lỗi 4: Khởi tạo `prev = root.val` hoặc `prev = 0`

`prev` phải là "nhỏ hơn mọi giá trị" (`-∞`). Nếu `prev = 0`, cây toàn số âm bị đánh giá sai.

In [19]:
def wrong_prev_zero(root: Optional[TreeNode]) -> bool:
    stack, node, prev = [], root, 0                     # SAI
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        if node.val <= prev:
            return False
        prev = node.val
        node = node.right
    return True

print("Sai :", wrong_prev_zero(build_tree([-2, -3, -1])), "| Đúng:", sol.isValidBST(build_tree([-2, -3, -1])))

Sai : False | Đúng: True


# 11. Mẹo nhận diện pattern

- **Ràng buộc truyền từ tổ tiên xuống** → DFS với tham số `(lo, hi)` (hoặc trạng thái khác).
- **Thứ tự sắp xếp trong BST** → inorder. Rất nhiều bài BST là "inorder + so với phần tử trước".

Bài liên quan: **#230 Kth Smallest Element in a BST** (inorder, dừng ở phần tử thứ `k`), **#99 Recover Binary Search Tree** (inorder tìm hai chỗ sai), **#501 Find Mode in BST**, **#530 Minimum Absolute Difference in BST**, **#700 Search in a BST**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Cây `[10, 5, 15, None, None, 6, 20]` có phải BST không? Nút nào vi phạm và vi phạm khoảng nào?

## Bài 2 — Điền chỗ trống (cách khoảng)

In [20]:
def exercise_2_is_valid_bst(root: Optional[TreeNode]) -> bool:
    def valid(node, lo, hi):
        # TODO
        pass
    return valid(root, float("-inf"), float("inf"))


# assert exercise_2_is_valid_bst(build_tree([2, 1, 3])) is True
# assert exercise_2_is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False
# assert exercise_2_is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7])) is False

## Bài 3 — Biến thể: Kth Smallest Element in a BST (LeetCode #230)

Trả về giá trị nhỏ thứ `k` (đếm từ 1) trong BST.

```python
kth_smallest([3, 1, 4, None, 2], 1)          -> 1
kth_smallest([5, 3, 6, 2, 4, None, None, 1], 3) -> 3
```

In [21]:
def exercise_3_kth_smallest(root: Optional[TreeNode], k: int) -> int:
    # TODO
    pass


# assert exercise_3_kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
# assert exercise_3_kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3

## Bài 4 — Biến thể: Minimum Absolute Difference in BST (LeetCode #530)

Hiệu nhỏ nhất giữa giá trị của **hai nút bất kỳ** trong BST.

```python
min_diff([4, 2, 6, 1, 3]) -> 1
min_diff([1, 0, 48, None, None, 12, 49]) -> 1
```

In [22]:
def exercise_4_min_diff(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_4_min_diff(build_tree([4, 2, 6, 1, 3])) == 1
# assert exercise_4_min_diff(build_tree([1, 0, 48, None, None, 12, 49])) == 1

## Bài 5 — Mini challenge: cây con BST lớn nhất (LeetCode #333)

Trong một cây nhị phân bất kỳ, tìm **số nút** của cây con lớn nhất là BST (cây con = một nút cùng **mọi** hậu duệ của nó). `O(n)`.

```python
largest_bst([10, 5, 15, 1, 8, None, 7]) -> 3     # cây con gốc 5: {1, 5, 8}
```

Gợi ý nhanh: DFS **hậu thứ tự** trả về `(là_BST, kích_thước, min, max)` của mỗi cây con.

In [23]:
def exercise_5_largest_bst(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_5_largest_bst(build_tree([10, 5, 15, 1, 8, None, 7])) == 3
# assert exercise_5_largest_bst(build_tree([2, 1, 3])) == 3
# assert exercise_5_largest_bst(build_tree([3, 2, 1])) == 1

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_range` sau khi tự làm.

**Bài 3**
- Mức 1: inorder bằng stack; đếm số nút đã thăm, dừng khi đếm bằng `k`.

**Bài 4**
- Mức 1: inorder cho dãy tăng → hiệu nhỏ nhất là giữa hai phần tử **liền kề** trong inorder.

**Bài 5**
- Mức 1: cây rỗng: `(True, 0, +∞, −∞)`.
- Mức 2: nút là gốc của BST nếu hai con đều BST và `left.max < val < right.min`; khi đó size = tổng + 1, min/max cập nhật.
- Mức 3: nếu không là BST, trả `(False, 0, −∞, +∞)` để mọi tổ tiên cũng không là BST; giữ biến toàn cục `best`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_range([10, 5, 15, None, None, 6, 20])

nút 10: cần trong (-inf, inf) -> OK
  nút 5: cần trong (-inf, 10) -> OK
  nút 15: cần trong (10, inf) -> OK
    nút 6: cần trong (10, 15) -> VI PHẠM
Kết quả: False


## Đáp án Bài 2

In [25]:
def answer_2_is_valid_bst(root: Optional[TreeNode]) -> bool:
    def valid(node, lo, hi):
        if not node:
            return True
        if not (lo < node.val < hi):
            return False
        return valid(node.left, lo, node.val) and valid(node.right, node.val, hi)
    return valid(root, float("-inf"), float("inf"))

assert answer_2_is_valid_bst(build_tree([2, 1, 3])) is True
assert answer_2_is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False
assert answer_2_is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7])) is False
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [26]:
def answer_3_kth_smallest(root: Optional[TreeNode], k: int) -> int:
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right
    raise ValueError("k lớn hơn số nút")

assert answer_3_kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
assert answer_3_kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_min_diff(root: Optional[TreeNode]) -> int:
    vals = inorder_list(root)
    return min(b - a for a, b in zip(vals, vals[1:]))

assert answer_4_min_diff(build_tree([4, 2, 6, 1, 3])) == 1
assert answer_4_min_diff(build_tree([1, 0, 48, None, None, 12, 49])) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_largest_bst(root: Optional[TreeNode]) -> int:
    best = 0
    def dfs(node):
        nonlocal best
        if not node:
            return True, 0, float("inf"), float("-inf")
        lb, ls, lmin, lmax = dfs(node.left)
        rb, rs, rmin, rmax = dfs(node.right)
        if lb and rb and lmax < node.val < rmin:
            size = ls + rs + 1
            best = max(best, size)
            return True, size, min(lmin, node.val), max(rmax, node.val)
        return False, 0, float("-inf"), float("inf")
    dfs(root)
    return best

assert answer_5_largest_bst(build_tree([10, 5, 15, 1, 8, None, 7])) == 3
assert answer_5_largest_bst(build_tree([2, 1, 3])) == 3
assert answer_5_largest_bst(build_tree([3, 2, 1])) == 1

def all_subtrees(root):
    return [] if not root else [root] + all_subtrees(root.left) + all_subtrees(root.right)

def size(root):
    return len(inorder_list(root))

for _ in range(2000):
    root = random_tree(random.randint(1, 8), 0, 6)
    expected = max(size(t) for t in all_subtrees(root) if brute_valid(t))
    assert answer_5_largest_bst(root) == expected
print("Bài 5 PASS (khớp brute force trên 2000 cây)")

Bài 5 PASS (khớp brute force trên 2000 cây)


# 15. Tổng kết

> BST là ràng buộc với **mọi tổ tiên**, không chỉ cha. Hai cách `O(n)`: truyền khoảng mở `(lo, hi)` xuống (bắt đầu `(-∞, +∞)`), hoặc kiểm tra inorder **tăng nghiêm ngặt**. Dùng stack để tránh đệ quy sâu.

---

## Checklist trước khi nộp

- [ ] Không chỉ so với con trực tiếp?
- [ ] Cận ban đầu là `±∞`, không phải `±2^31`?
- [ ] So sánh **nghiêm ngặt** (`<`, không `<=`)?
- [ ] `prev` khởi tạo `-∞`?